# Identification seed point par FTD

In [ ]:
# Chargement des librairies
import numpy as np
import matplotlib.pyplot as plt
import os
import plotly.graph_objects as go
import nibabel as nib
from skimage.feature import canny
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.draw import circle_perimeter
from scipy.signal import convolve2d
from skimage.segmentation import flood, flood_fill


In [ ]:
# Let's start with a single image
# Visualisation des images d'un patient

training_path = "database/training"

# Visualisation des images d'un patient
numero_patient = 1

# Load the images
image_4d = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_4d.nii.gz")
image_01 = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame01.nii.gz")
image_01_gt = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame01_gt.nii.gz")
image_12 = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame12.nii.gz")
image_12_gt = nib.load(f"{training_path}/patient{numero_patient:03d}/patient{numero_patient:03d}_frame12_gt.nii.gz")

print(image_4d.shape) # (216, 256, 10, 30) for patient 1, corresponding to 10 slices and 30 time frames
r, c, nb_frames, duration = image_4d.shape

print("Images loaded")

In [ ]:
# Plot the images

layer = 4

fig, axs = plt.subplots(2, 2, figsize=(10, 10))
axs[0, 0].imshow(image_01.dataobj[:, :, layer], cmap="gray")
axs[0, 0].set_title(f"Frame 01, layer {layer}")
axs[1, 0].imshow(image_01_gt.dataobj[:, :, layer], cmap="gray")
axs[1, 0].set_title(f"Frame 01 GT, layer {layer}")
axs[0, 1].imshow(image_12.dataobj[:, :, layer], cmap="gray")
axs[0, 1].set_title(f"Frame 12, layer {layer}")
axs[1, 1].imshow(image_12_gt.dataobj[:, :, layer], cmap="gray")
axs[1, 1].set_title(f"Frame 12 GT, layer {layer}")
plt.show()


In [ ]:
# Let's perform a fourrier transform on the first image (time axis)
image = image_4d.get_fdata()

# Perform the fourrier transform
image_fft = np.fft.fft(image, axis=3)

print(image_fft.shape)

# Plot the fourrier transform for layer 4
fig, axs = plt.subplots(1, 10, figsize=(20, 2))
for i in range(10):
    axs[i].imshow(np.abs(image_fft[:, :, 4, i]), cmap="gray")
    axs[i].set_title(f"Frame {i} (layer 4)")
plt.show()

# Only keep the first harmonic
image_fft_h1 = abs(image_fft[:, :, :, 1])

print(image_fft_h1.shape)

# Keep the highest values
image_fft_h1 = image_fft_h1 > np.percentile(image_fft_h1, 95)

# Plot the average fourrier transform
plt.imshow(image_fft_h1[:, :, 0], cmap="gray")
plt.title("Fourrier transform filter (threshold 95%)")
plt.show()

In [ ]:
# Perform the hough transform and plot all found circles.

# Edge detection for each layer
edges = np.zeros(image_fft_h1.shape)
for i in range(image_fft_h1.shape[2]):
    edges[:, :, i] = canny(image_fft_h1[:, :, i], sigma=0.3, low_threshold=0.0001, high_threshold=1)

fig, axs = plt.subplots(1, 5, figsize=(20, 2))
for j in range(5):
    axs[j].imshow(edges[:, :, j], cmap="gray")
    axs[j].set_title(f"Layer {j}")
plt.show()

# Hough transform for each layer
centers = []
for i in range(image_fft_h1.shape[2]):
    hough_radii = np.arange(10, 20, 1)
    hough_res = hough_circle(edges[:, :, i], hough_radii)
    accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=1)

    # Compute the average of the centers
    cx = int(np.mean(cx))
    cy = int(np.mean(cy))

    centers.append((cx, cy))

avg_center = np.mean(centers, axis=0)
print(f"Average center: {avg_center}")

# Plot the center point on the original image (layer 4 )
fig, axs = plt.subplots(1, 5, figsize=(20, 2))
for i in range(5):
    axs[i].imshow(image[:, :, i, 4], cmap="gray")
    axs[i].scatter(avg_center[0], avg_center[1], c="red")
    axs[i].set_title(f"Layer {i}")
plt.show()

In [ ]:
def compute_fft_avg(image_4d):
    # Perform the fourrier transform on the first image (time axis)
    image = image_4d.get_fdata()

    # Perform the fourrier transform
    image_fft = np.fft.fft(image, axis=3)

    # Only keep the first harmonic
    image_fft_h1 = abs(image_fft[:, :, :, 1])

    # Keep the highest values
    image_fft_h1 = image_fft_h1 > np.percentile(image_fft_h1, 95)

    return image_fft_h1

def compute_center(image_fft_h1):
    # Perform the hough transform and plot all found circles.

    # Edge detection for each layer
    edges = np.zeros(image_fft_h1.shape)
    for i in range(image_fft_h1.shape[2]):
        edges[:, :, i] = canny(image_fft_h1[:, :, i], sigma=1, low_threshold=0.1, high_threshold=1)

    # Hough transform for each layer
    centers = []
    for i in range(image_fft_h1.shape[2]):
        hough_radii = np.arange(10, 20, 1)
        hough_res = hough_circle(edges[:, :, i], hough_radii)
        accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=1)

        # Compute the average of the centers
        cx = int(np.mean(cx))
        cy = int(np.mean(cy))

        centers.append((cx, cy))

    avg_center = np.mean(centers, axis=0)

    avg_center = (int(avg_center[0]), int(avg_center[1]))

    return avg_center

print("Functions defined")

In [ ]:
# Let's test with a few patients
patient_numbers = [i for i in range(1, 20)]

for patient in patient_numbers:
    image_4d = nib.load(f"{training_path}/patient{patient:03d}/patient{patient:03d}_4d.nii.gz")
    image_fft_h1 = compute_fft_avg(image_4d)
    center = compute_center(image_fft_h1)
    print(f"Patient {patient:03d}: Center {center}")

    # Parameters for visualization
    layer, timeframe = 4, 4
    plt.imshow(image_4d.dataobj[:, :, layer, timeframe], cmap="gray")
    plt.scatter(center[0], center[1], c="red")
    plt.title(f"Patient {patient:03d}, layer {layer}, frame {timeframe}")
    plt.show()

In [ ]:
# Let's try to do region growing to segment the right ventricle.

# Let's test with a few patients
patient_numbers = [i for i in range(1, 5)]

for patient in patient_numbers:
    image_4d = nib.load(f"{training_path}/patient{patient:03d}/patient{patient:03d}_4d.nii.gz")
    image_fft_h1 = compute_fft_avg(image_4d)
    center = compute_center(image_fft_h1)
    print(f"Patient {patient:03d}: Center {center}")    
    
    # Only keep a small region around the center (square of 100x100)
    x, y, _, _ = image_4d.shape
    x_min, x_max = max(center[0] - 64, 0), min(center[0] + 64, x)
    y_min, y_max = max(center[1] - 64, 0), min(center[1] + 64, y)
    print(f"Region: ({x_min}, {x_max}), ({y_min}, {y_max})")
    image_center = image_4d.dataobj[y_min:y_max, x_min:x_max, :, :]

    # Visualize the region
    fig, axs = plt.subplots(1, 5, figsize=(20, 2))
    for i in range(5):
        axs[i].imshow(image_center[:, :, i, 4], cmap="gray")
        axs[i].scatter(center[0] - x_min, center[1] - y_min, c="red")
        axs[i].set_title(f"Layer {i}")
    plt.show()

    # Perform region growing
    fig, axs = plt.subplots(1, 5, figsize=(20, 2))
    for i in range(5):
        seed = (center[1] - y_min, center[0] - x_min)
        seed_intensity = image_center[seed[0], seed[1], i, 4]
        mask = flood(image_center[:, :, i, 4], seed, tolerance=0.1*seed_intensity)
        axs[i].imshow(mask, cmap="gray")
        axs[i].scatter(center[0] - x_min, center[1] - y_min, c="red")
        axs[i].set_title(f"Layer {i}")
    plt.show()

    # Make the mask appear on the original image
    fig, axs = plt.subplots(1, 5, figsize=(20, 2))
    for i in range(5):
        seed = (center[1] - y_min, center[0] - x_min)
        seed_intensity = image_center[seed[0], seed[1], i, 4]
        mask = flood(image_center[:, :, i, 4], seed, tolerance=0.1*seed_intensity)
        axs[i].imshow(image_center[:, :, i, 4], cmap="gray")
        axs[i].imshow(mask, cmap="jet", alpha=0.5)
        axs[i].scatter(center[0] - x_min, center[1] - y_min, c="red")
        axs[i].set_title(f"Layer {i}")
    plt.show()
